<a href="https://colab.research.google.com/github/nikenyudha/Learn_SQL/blob/main/Learn_SQL_BASIC_!!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##USE SQLLITE LIBRARY AND TEST

In [ ]:
# Import library
import sqlite3

# Connect to an SQLite database; use ':memory:' for an in-memory database
conn = sqlite3.connect(':memory:')


In [ ]:
# Execute a SQL command to create a new table
c = conn.cursor()
c.execute('''
          CREATE TABLE Students
          (name text, age real, weight real)
          ''')


In [ ]:
# Execute a SQL command to insert data into the table
c.execute("INSERT INTO Students VALUES ('John',28,70)")

# Commit the transaction to save changes to the database
conn.commit()


In [ ]:
# Execute a SQL SELECT statement to query the database
c.execute("SELECT * FROM Students")

# Fetch all rows from the result of the query
print(c.fetchall())

[('John', 28.0, 70.0)]


## String Function

### UPPER(), LOWER(), LENGHT()

In [ ]:
c = conn.cursor()
c.execute('''
          CREATE TABLE contacts
          (id INTEGER PRIMARY KEY,
           first_name TEXT,
           last_name TEXT,
           email TEXT)
          ''')

In [ ]:
c.execute("INSERT INTO contacts VALUES (1, 'alice', 'JOHNSON', 'alice@example.com')")
c.execute("INSERT INTO contacts VALUES (2, 'bob', 'SMITH', 'bob@example.com')")
c.execute("INSERT INTO contacts VALUES (3, 'Carol', 'Williams', 'carolharlie@example.com')")
c.execute("INSERT INTO contacts VALUES (4, 'David', 'Lee', 'david@example.com')")

conn.commit()

In [ ]:
c.execute("SELECT id, UPPER(first_name) AS first_upper, LOWER(last_name) AS last_lower, LOWER(email) AS normalized_email, LENGTH(email) AS email_length FROM contacts")
print(c.fetchall())


[(1, 'ALICE', 'johnson', 'alice@example.com', 17), (2, 'BOB', 'smith', 'bob@example.com', 15), (3, 'CAROL', 'williams', 'carolharlie@example.com', 23), (4, 'DAVID', 'lee', 'david@example.com', 17)]


In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT id, UPPER(first_name) AS first_upper, LOWER(last_name) AS last_lower, LOWER(email) AS normalized_email, LENGTH(email) AS email_length FROM contacts", conn)
print(df)

   id first_upper last_lower         normalized_email  email_length
0   1       ALICE    johnson        alice@example.com            17
1   2         BOB      smith          bob@example.com            15
2   3       CAROL   williams  carolharlie@example.com            23
3   4       DAVID        lee        david@example.com            17


### SUBSTR() INSTR()
`SUBSTR`(string, start, length) takes three arguments: the string, the 1-based start index, and an optional length. If you omit the length, it returns everything from the start position to the end of the string.
`INSTR`(string, pattern) returns the position of the first match, or 0 if the pattern isn't found.

In [ ]:
conn.cursor()
c.execute('''
          CREATE TABLE products
          (id INTEGER PRIMARY KEY,
           code TEXT NOT NULL,
           name TEXT NOT NULL,
           price REAL NOT NULL)
          ''')

In [ ]:
c.execute("INSERT OR IGNORE INTO products VALUES (1, 'ELC-MON-0042', 'Monitor 24', 349.99)")
c.execute("INSERT OR IGNORE INTO products VALUES (2, 'ELC-KBD-0117', 'Mechanical Keyboard', 89.99)")
c.execute("INSERT OR IGNORE INTO products VALUES (3, 'FRN-CHR-0008', 'Ergonomic Chair', 499.00)")
c.execute("INSERT OR IGNORE INTO products VALUES (4, 'FRN-DSK-0031', 'Standing Desk', 799.00)")
c.execute("INSERT OR IGNORE INTO products VALUES (5, 'ELC-MSE-0095', 'Wireless Mouse', 44.99)")

conn.commit()

In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT code, name, SUBSTR(code, 1, 3) AS category, SUBSTR(code, 5, 3) AS subcategory, SUBSTR(code, 9) AS item_number, INSTR(code, '-') AS first_dash_position FROM products", conn)
print(df)

           code                 name category subcategory item_number  \
0  ELC-MON-0042           Monitor 24      ELC         MON        0042   
1  ELC-KBD-0117  Mechanical Keyboard      ELC         KBD        0117   
2  FRN-CHR-0008      Ergonomic Chair      FRN         CHR        0008   
3  FRN-DSK-0031        Standing Desk      FRN         DSK        0031   
4  ELC-MSE-0095       Wireless Mouse      ELC         MSE        0095   

   first_dash_position  
0                    4  
1                    4  
2                    4  
3                    4  
4                    4  


### TRIM and REPLACE

`TRIM()` without a second argument removes whitespace. You can also pass a specific character to trim: TRIM(name, '.') removes leading and trailing periods. `REPLACE`(string, old, new) replaces every match — notice how nesting multiple REPLACE() calls lets you strip several different characters in one pass.

In [ ]:
conn.cursor()
c.execute('''
          CREATE TABLE raw_employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    phone TEXT,
    department TEXT)
          ''')

In [ ]:
c.execute("INSERT OR IGNORE INTO raw_employees VALUES (1, '  Sarah Connor  ', '555-867-5309', '  Engineering  ')")
c.execute("INSERT OR IGNORE INTO raw_employees VALUES (2, 'John   Doe', '(555) 234-5678', 'Marketing')")
c.execute("INSERT OR IGNORE INTO raw_employees VALUES (3, '  Maria Garcia', '555.901.2345', 'Engineering  ')")
c.execute("INSERT OR IGNORE INTO raw_employees VALUES (4, 'James Wilson  ', '555-444-9988', '  HR')")

conn.commit()

In [ ]:
import pandas as pd
query = """
SELECT
    id,
    TRIM(name) AS clean_name,
    TRIM(department) AS clean_department,
    -- Remove all non-digit formatting from phone numbers
    REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(phone, '-', ''), ' ', ''), '(', ''), ')', ''), '.', '') AS digits_only,
    LENGTH(TRIM(name)) AS name_length
FROM raw_employees
"""

df_employess_trim_clean = pd.read_sql_query(query, conn)
print(df_employess_trim_clean)

   id    clean_name clean_department digits_only  name_length
0   1  Sarah Connor      Engineering  5558675309           12
1   2    John   Doe        Marketing  5552345678           10
2   3  Maria Garcia      Engineering  5559012345           12
3   4  James Wilson               HR  5554449988           12


### combining string and concat

In [ ]:
conn.cursor()
c.execute('''
          CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    department TEXT NOT NULL,
    hire_year INTEGER NOT NULL)
          ''')

In [ ]:
c.execute("INSERT OR IGNORE INTO employees VALUES (1, 'Alice', 'Johnson', 'Engineering', 2019)")
c.execute("INSERT OR IGNORE INTO employees VALUES (2, 'Bob', 'Smith', 'Marketing', 2021)")
c.execute("INSERT OR IGNORE INTO employees VALUES (3, 'Carol', 'Williams', 'Engineering', 2018)")
c.execute("INSERT OR IGNORE INTO employees VALUES (4,  'David', 'Brown', 'HR', 2022)")

conn.commit()

In [ ]:
import pandas as pd
df_employees = pd.read_sql_query("SELECT first_name || ' ' || last_name AS full_name, UPPER(SUBSTR(first_name,1, 1)) || '.' || last_name AS formal_name, LOWER(first_name) || '.' || LOWER(last_name) || '@company.com' AS email, 'Joined' || hire_year || '(' || department || ')' AS summary FROM employees ORDER BY last_name", conn)
print(df_employees)

        full_name formal_name                       email  \
0     David Brown     D.Brown     david.brown@company.com   
1   Alice Johnson   A.Johnson   alice.johnson@company.com   
2       Bob Smith     B.Smith       bob.smith@company.com   
3  Carol Williams  C.Williams  carol.williams@company.com   

                   summary  
0           Joined2022(HR)  
1  Joined2019(Engineering)  
2    Joined2021(Marketing)  
3  Joined2018(Engineering)  


In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT first_name || ' ' || last_name AS full_name, UPPER(SUBSTR(first_name,1, 1)) || '.' || last_name AS formal_name, LOWER(first_name) || '.' || LOWER(last_name) || '@company.com' AS email, 'Joined' || hire_year || '(' || department || ')' AS summary FROM employees ORDER BY last_name", conn)
print(df)

        full_name formal_name                       email  \
0     David Brown     D.Brown     david.brown@company.com   
1   Alice Johnson   A.Johnson   alice.johnson@company.com   
2       Bob Smith     B.Smith       bob.smith@company.com   
3  Carol Williams  C.Williams  carol.williams@company.com   

                   summary  
0           Joined2022(HR)  
1  Joined2019(Engineering)  
2    Joined2021(Marketing)  
3  Joined2018(Engineering)  


### Exercise

In [ ]:
conn.cursor()
c.execute('''
          CREATE TABLE IF NOT EXISTS purchases (
      id INTEGER PRIMARY KEY,
      customer TEXT,
      product_code TEXT,
      amount REAL)
   ''')

In [ ]:
c.execute("INSERT OR IGNORE INTO purchases VALUES (1, 'alice JOHNSON', '  elc-mon-0042  ', 349.99)")
c.execute("INSERT OR IGNORE INTO purchases VALUES (2, 'BOB smith', 'ELC-KBD-0117', 89.99)")
c.execute("INSERT OR IGNORE INTO purchases VALUES (3, 'carol williams', '  FRN-CHR-0008', 499.00)")
c.execute("INSERT OR IGNORE INTO purchases VALUES (4, 'DAVID brown  ', 'frn-dsk-0031  ', 799.00)")
c.execute("INSERT OR IGNORE INTO purchases VALUES (5, 'Eva Martinez', '  ELC-MSE-0095  ', 44.99)")

conn.commit()

In [ ]:
import pandas as pd
dfraw= pd.read_sql_query("SELECT * FROM purchases", conn)
print(dfraw);

   id        customer      product_code  amount
0   1   alice JOHNSON    elc-mon-0042    349.99
1   2       BOB smith      ELC-KBD-0117   89.99
2   3  carol williams      FRN-CHR-0008  499.00
3   4   DAVID brown      frn-dsk-0031    799.00
4   5    Eva Martinez    ELC-MSE-0095     44.99


start here

In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT id, TRIM(product_code) AS clean_code FROM purchases", conn)
print(df);

   id    clean_code
0   1  elc-mon-0042
1   2  ELC-KBD-0117
2   3  FRN-CHR-0008
3   4  frn-dsk-0031
4   5  ELC-MSE-0095


- Challenge 1: Display customer names in Title Case (first letter upper, rest lower)
-- Hint: combine UPPER, LOWER, SUBSTR, and INSTR to find the space between names

In [ ]:
query = """
SELECT id,
       UPPER(SUBSTR(customer, 1, 1)) ||
       LOWER(SUBSTR(customer, 2, INSTR(customer, ' ') - 2)) ||
       ' ' ||
       UPPER(SUBSTR(customer, INSTR(customer, ' ') + 1, 1)) ||
       LOWER(SUBSTR(customer, INSTR(customer, ' ') + 2)) AS name_title_case
FROM purchases
"""

df2 = pd.read_sql_query(query, conn)
print(df2)

   id name_title_case
0   1   Alice Johnson
1   2       Bob Smith
2   3  Carol Williams
3   4   David Brown  
4   5    Eva Martinez


In [ ]:
import pandas as pd
df1 = pd.read_sql_query("SELECT id, TRIM(UPPER(SUBSTR(customer, 1, 1)) || LOWER(SUBSTR(customer, 2))) AS title_case_name FROM purchases", conn)
print(df1)

   id title_case_name
0   1   Alice johnson
1   2       Bob smith
2   3  Carol williams
3   4     David brown
4   5    Eva martinez


-- Challenge 2: Normalize product codes (trim whitespace, uppercase)
-- Then extract just the category prefix (first 3 characters)

In [ ]:
import pandas as pd

query = """
SELECT product_code,
       TRIM(UPPER(product_code)) AS normalized_code,
       SUBSTR(TRIM(UPPER(product_code)), 1, 3) AS category_prefix,
       SUBSTR(TRIM(UPPER(product_code)), 5, 3) AS subcategory_prefix,
       SUBSTR(TRIM(UPPER(product_code)), 9) AS item_number
FROM purchases
"""

dfproduct_codes = pd.read_sql_query(query, conn)
print(dfproduct_codes)

       product_code normalized_code category_prefix subcategory_prefix  \
0    elc-mon-0042      ELC-MON-0042             ELC                MON   
1      ELC-KBD-0117    ELC-KBD-0117             ELC                KBD   
2      FRN-CHR-0008    FRN-CHR-0008             FRN                CHR   
3    frn-dsk-0031      FRN-DSK-0031             FRN                DSK   
4    ELC-MSE-0095      ELC-MSE-0095             ELC                MSE   

  item_number  
0        0042  
1        0117  
2        0008  
3        0031  
4        0095  


-- Challenge 3: Build a receipt line like:
-- "Alice Johnson purchased ELC-MON-0042 for $349.99"

In [ ]:
import pandas as pd

query = """
SELECT customer || ' purchased ' || product_code || ' for $' || amount AS receipt_line
FROM purchases
"""

dfreceipt_line = pd.read_sql_query(query, conn)
print(dfreceipt_line)

                                        receipt_line
0  alice JOHNSON purchased   elc-mon-0042   for $...
1        BOB smith purchased ELC-KBD-0117 for $89.99
2  carol williams purchased   FRN-CHR-0008 for $4...
3  DAVID brown   purchased frn-dsk-0031   for $799.0
4  Eva Martinez purchased   ELC-MSE-0095   for $4...


In [ ]:
#Ambil kolom dari Masing masing DF
import pandas as pd
#switch to string
amount_str = dfraw['amount'].astype(str)

receipt_line = (df2['name_title_case'] + " purchased " + dfproduct_codes['normalized_code'] + " for $" + amount_str)

df_final = pd.DataFrame({'receipt_line': receipt_line})
print(df_final)


                                       receipt_line
0  Alice Johnson purchased ELC-MON-0042 for $349.99
1       Bob Smith purchased ELC-KBD-0117 for $89.99
2  Carol Williams purchased FRN-CHR-0008 for $499.0
3   David Brown   purchased FRN-DSK-0031 for $799.0
4    Eva Martinez purchased ELC-MSE-0095 for $44.99


In [ ]:
#use concat (digabung semuanya dulu)
import pandas as pd

# Satukan kolom-kolom yang dibutuhkan dari df yang berbeda secara berdampingan (axis=1)
df_gabung = pd.concat([
    df2['name_title_case'],
    dfproduct_codes['normalized_code'],
    dfraw['amount'] # Sesuaikan df asal kolom amount-mu
], axis=1)

# 2. Ubah tipe data amount yang REAL/Float menjadi String
df_gabung['amount'] = df_gabung['amount'].astype(str)

#3 Format kolom REAL/Float agar selalu menampilkan 2 angka di belakang koma
#df_gabung['amount'] = df_gabung['amount'].map('{:.2f}'.format)

# 3. Kalimat struk belandanya
df_gabung['receipt_line'] = (
    df_gabung['name_title_case'] +
    ' purchased ' +
    df_gabung['normalized_code'] +
    ' for $' +
    df_gabung['amount']
)

# Tampilan hasil akhir
print(df_gabung[['receipt_line']])

                                       receipt_line
0  Alice Johnson purchased ELC-MON-0042 for $349.99
1       Bob Smith purchased ELC-KBD-0117 for $89.99
2  Carol Williams purchased FRN-CHR-0008 for $499.0
3   David Brown   purchased FRN-DSK-0031 for $799.0
4    Eva Martinez purchased ELC-MSE-0095 for $44.99
